<a href="https://colab.research.google.com/github/MaiAlhusseini/FlyRank_Intern_repo/blob/main/Copy_of_w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaiAlhusseini/FlyRank_Intern_repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method Choice and Why

For Lane 2, the goal is to prioritize pages for refresh review.

I use a Random Forest classifier to estimate whether a page has the measured declining trend label. It can combine content, visibility, freshness, search, and engagement signals, including non-linear relationships.

The result is decision support, not proof that refreshing a page will improve performance. I evaluate the ranking using Precision@50 because the content team would review only a limited number of highest-priority pages.

## 2. Split Design

I use a client-grouped split: all pages from a client are placed in either training or test data, never both. This is more honest than a row-level random split because it tests whether the model generalizes to unseen clients.

The Week 4 notebook ranked the full dataset and did not create an evaluation split. Therefore, I apply this same Week 5 client-held-out split to both the leakage-safe baseline and the model.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from google.colab import files

uploaded = files.upload()

import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")
baseline = pd.read_csv("baseline_action_score.csv")

# Use client_id to keep the same client's pages together
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print(
    "Clients appearing in both:",
    len(
        set(train_df["client_id"])
        & set(test_df["client_id"])
    )
)

Saving baseline_action_score.csv to baseline_action_score (3).csv
Saving content_refresh_anonymized.csv to content_refresh_anonymized (5).csv
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Clients appearing in both: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Target: measured declining trend
train_df = train_df[train_df["trend_direction"].notna()].copy()
test_df = test_df[test_df["trend_direction"].notna()].copy()

y_train = train_df["trend_direction"].eq("down").astype(int)
y_test = test_df["trend_direction"].eq("down").astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
feature_columns = numeric_features + categorical_features

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
model_probability = model.predict_proba(X_test)[:, 1]


def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).rank(pct=True)

def safe_baseline_score(frame):
    impressions = pd.to_numeric(frame["impressions_90d"], errors="coerce").fillna(0)
    freshness = pd.to_numeric(frame["days_since_last_update"], errors="coerce").fillna(0)
    position = pd.to_numeric(frame["avg_position"], errors="coerce").fillna(0)
    words = pd.to_numeric(frame["word_count"], errors="coerce").fillna(0)

    visibility_score = percentile_rank(np.log1p(impressions))
    freshness_risk_score = percentile_rank(freshness)

    position_opportunity_score = (
        (1 - ((position.clip(lower=1, upper=50) - 1) / 49))
        * visibility_score
        * (position > 0).astype(int)
    )

    depth_gap_score = (1 - percentile_rank(words)) * visibility_score

    return (
        0.40 * visibility_score
        + 0.30 * freshness_risk_score
        + 0.25 * position_opportunity_score
        + 0.05 * depth_gap_score
    )

baseline_score = safe_baseline_score(test_df)

def precision_at_k(y_true, scores, k=50):
    ranking = pd.DataFrame({
        "actual_declining": np.asarray(y_true),
        "score": np.asarray(scores)
    }).sort_values("score", ascending=False)

    return ranking.head(min(k, len(ranking)))["actual_declining"].mean()

model_pred = (model_probability >= 0.5).astype(int)

comparison = pd.DataFrame({
    "method": ["Leakage-safe Week-4-style baseline", "Random Forest"],
    "precision_at_50": [
        precision_at_k(y_test, baseline_score, 50),
        precision_at_k(y_test, model_probability, 50)
    ],
    "precision_at_100": [
        precision_at_k(y_test, baseline_score, 100),
        precision_at_k(y_test, model_probability, 100)
    ],
    "precision_at_0_5": [
        precision_score(y_test, (baseline_score >= 0.5).astype(int), zero_division=0),
        precision_score(y_test, model_pred, zero_division=0)
    ],
    "recall_at_0_5": [
        recall_score(y_test, (baseline_score >= 0.5).astype(int), zero_division=0),
        recall_score(y_test, model_pred, zero_division=0)
    ],
    "f1_at_0_5": [
        f1_score(y_test, (baseline_score >= 0.5).astype(int), zero_division=0),
        f1_score(y_test, model_pred, zero_division=0)
    ]
})

print("Target rate in held-out clients:", round(y_test.mean(), 3))
print("Features used:", len(feature_columns))

comparison



Target rate in held-out clients: 0.511
Features used: 30


,method,precision_at_50,precision_at_100,precision_at_0_5,recall_at_0_5,f1_at_0_5
0,Leakage-safe Week-4-style baseline,0.32,0.32,0.474788,0.337885,0.394805
1,Random Forest,0.60,0.57,0.594838,0.607494,0.601100


The client-held-out test set had a declining-label rate of 51.1%. Using the same held-out clients and Precision@50, the leakage-safe Week-4-style baseline achieved 0.32, while the Random Forest achieved 0.60.

This means that among the 50 pages prioritized by each method, the baseline identified about 16 declining pages and the Random Forest identified about 30. The Random Forest therefore identified 14 more declining pages within the same review capacity.

The Random Forest also performed better at Precision@100 (0.57 versus 0.32). I therefore select the Random Forest as the stronger ranking method for this experiment. This result is decision support based on observed associations; it does not prove that refreshing a recommended page will improve its performance.

In [ ]:
historical_baseline = baseline.set_index("content_id")["baseline_score"]
historical_score_on_test = test_df["content_id"].map(historical_baseline)

print(
    "Original Week 4 score available for test pages:",
    historical_score_on_test.notna().mean()
)

Original Week 4 score available for test pages: 1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# Feature importance
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

print("Top model signals:")
display(importance.head(10))

# Review the model's top-50 recommendations
review = test_df[[
    "content_id", "client_id", "content_age_days",
    "days_since_last_update", "impressions_90d", "avg_position", "word_count"
]].copy()

review["actual_declining"] = y_test.to_numpy()
review["model_probability"] = model_probability
review["baseline_score"] = baseline_score.to_numpy()

top50_model = review.sort_values("model_probability", ascending=False).head(50)

false_positive_picks = top50_model[
    top50_model["actual_declining"] == 0
]

missed_declining_pages = review[
    review["actual_declining"] == 1
].sort_values("model_probability").head(10)

print("Model top-50 pages that were not labelled declining:")
display(false_positive_picks.head(10))

print("Declining pages the model gave the lowest priority:")
display(missed_declining_pages)

Top model signals:


,feature,importance
13,numeric__days_with_impressions,0.157609
5,numeric__impressions_90d,0.113343
18,numeric__avg_position,0.102719
15,numeric__content_age_days,0.099020
4,numeric__char_count,0.038974
3,numeric__word_count,0.033708
17,numeric__ctr,0.029513
34,categorical__age_tier_365+,0.028774
20,numeric__scroll_rate,0.028742
6,numeric__clicks_90d,0.024967


Model top-50 pages that were not labelled declining:


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,word_count,actual_declining,model_probability,baseline_score
10080,content_35d63627bf3e,client_8527a891e2,238,103,1525,32.6,1592.0,0,0.859718,0.608341
22526,content_1d0963b56227,client_4e07408562,280,104,3445,39.0,1480.0,0,0.847850,0.671199
11061,content_0b47dae0c7f9,client_8527a891e2,238,103,1191,23.1,1514.0,0,0.846280,0.620775
22524,content_846bb4dd8b44,client_8527a891e2,275,104,870,17.6,1492.0,0,0.837510,0.634970
22042,content_2ba626fea4d6,client_8527a891e2,275,104,360,7.2,1405.0,0,0.835389,0.590712
13,content_a5a2fbc76336,client_8527a891e2,238,103,307,39.8,1342.0,0,0.832798,0.478073
4050,content_500bd3907331,client_4e07408562,230,104,4037,5.5,1294.0,0,0.832208,0.822457
29456,content_b46c62b14582,client_8527a891e2,238,103,6240,31.8,1312.0,0,0.832159,0.712888
5477,content_3164f3076003,client_8527a891e2,275,104,2696,16.1,1274.0,0,0.829478,0.745156
12069,content_ff4370afd49c,client_4e07408562,280,104,1677,33.1,1516.0,0,0.827734,0.639432


Declining pages the model gave the lowest priority:


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,word_count,actual_declining,model_probability,baseline_score
1864,content_16f38acf0f26,client_e629fa6598,358,20,2,50.0,1590.0,1,0.108585,0.154597
27271,content_7bc32bc1df59,client_8527a891e2,238,92,1,0.0,1429.0,1,0.114474,0.249105
2936,content_064539f383aa,client_8527a891e2,277,20,2,5.5,3653.0,1,0.138230,0.162011
19104,content_a860ee9e7ae4,client_8527a891e2,127,104,1,8.0,4049.0,1,0.150472,0.284231
18423,content_77e2a54525b6,client_8527a891e2,273,20,1,7.0,1449.0,1,0.151091,0.146529
8441,content_9fb007756e5a,client_8527a891e2,310,20,2,20.5,3864.0,1,0.151135,0.159005
13114,content_8f222654e93f,client_8527a891e2,273,20,2,5.5,1994.0,1,0.151150,0.163262
25350,content_a4c38287770e,client_8527a891e2,275,20,2,5.0,1495.0,1,0.151675,0.163523
27395,content_e72e6c56f0a3,client_8527a891e2,273,20,1,9.0,1320.0,1,0.156724,0.146408
28072,content_2847e276c475,client_8527a891e2,271,20,1,6.0,1464.0,1,0.157218,0.146595


## 4. Errors and Interpretation

The primary ranking metric was Precision@50. On held-out clients, the leakage-safe baseline achieved [0.32] and the Random Forest achieved [0.60].

The model's top signals were [numeric__days_with_impressions	, numeric__impressions_90d	 , numeric__avg_position , numeric__content_age_days	 , numeric__char_count , numeric__word_count	 , numeric__ctr	 , categorical__age_tier_365+	 , numeric__scroll_rate	, numeric__clicks_90d	]. These are associations the model used, not evidence that those signals cause decline or prove that a refresh will work.

I reviewed false-positive top-50 recommendations and declining pages that received low model scores. A false positive may reflect a page that is old or visible but not currently declining. A false negative may reflect decline caused by information not represented in the available features.

The original Week 4 rule used current `trend_pct`, which directly relates to the decline label. I therefore treated it as a transparent descriptive triage rule and used a leakage-safe version for the model comparison.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.